Creates a very basic de novo simulated dataset, with a more realistic ratio of true positive to true negative than `simple_spread` provides.

This is similar to `simple_de_novo_simulation.ipynb`

# Imports

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd
 

%load_ext autoreload
%autoreload 2

2025-09-12 18:16:42.251570: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-12 18:16:42.255675: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

# Create cluster

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='24GB')
client=Client(cluster)

# Seting experimiental design parameters

In [3]:
#first, we define the new parameters we want to assign to this object.

new_cell_number=pd.Series({"reference":1000,"blood":2000,"neuron":1000})

#we make up 3x replicates
new_zi=pd.Series({"replicate_A":0.02,"replicate_B":0.021})

new_min=1
new_max=200

new_MOI=60

In [4]:
#next, let's create a bounds object from these parameters and the bounds of the shendure data.
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi,
    cells_per_cell_type=new_cell_number)
    
artificial_bounds.set_effective_moi(new_MOI)

# Creating an artificial library

`cat shendure_counts_grouped.txt | cut -f4,11 | grep "^minP" | cut -f2 | awk '{ sum += $1; n++ } END { if (n > 0) print sum / n }'`
Produces `0.0727759` as the average MPRA UMIs per cell for minP.

In [5]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(new_cell_number.keys()),
    minimum=new_min,
    maximum=new_max,
    minp_value=0.0727759,
    total=1000,
    frac_active=0.5,
    ct_specificity=.2)

scMPRAforge: INFO: 85.6% of active elements are not cell-type specific.
